# Main実験
pre-main共通エンジンによるA/B/F学習とA〜F評価。設定・続き方はREADME参照。実行flagの初期値はFalse。

In [ ]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
import sys
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from uav_safe_marl.experiments.main import prepare, train_method, evaluate_condition, context
from uav_safe_marl.experiments.main_analysis import analyze
OUTPUT = ROOT / 'results' / 'main_four_faces_free_30m'
SETTINGS = ROOT / 'notebooks' / 'main' / 'settings.yaml'
print(OUTPUT)


## 固定bankの準備
設定を変える場合は別OUTPUTにする。確認Testは生成のみで評価しない。

In [ ]:
RUN_PREPARE = False
if RUN_PREPARE:
    experiment = prepare(SETTINGS, OUTPUT)
    print(experiment['bank_hashes'])


## 学習と継続
各seedでA/B/Fをそろえる。C/Dを学習しない。初回は全9 runを30kで停止。developmentで延長が必要と判断した場合だけ、全9 runを同じ出力先・方式・seedでTARGET=60000にする。

In [ ]:
RUN_TRAIN = False
METHOD = 'A'
SEED = 0
TARGET = 30000
if RUN_TRAIN:
    print(train_method(OUTPUT, METHOD, SEED, TARGET))


## 判断用評価
A/B/C/D/F/F_OFFは各seed、E/Rは1回。

In [ ]:
RUN_EVALUATION = False
METHODS = ['A', 'B', 'C', 'D', 'E', 'F', 'R', 'F_OFF']
if RUN_EVALUATION:
    for method in METHODS:
        seeds = [0] if method in ('E', 'R') else context(OUTPUT)['settings']['training_seeds']
        for seed in seeds:
            evaluate_condition(OUTPUT, method, seed, checkpoint_step=TARGET)


## 確率方策校正
HOCBF ON・同じ初期sample actionで校正する。

In [ ]:
RUN_CALIBRATION = False
if RUN_CALIBRATION:
    for method in ['A', 'B', 'F']:
        for seed in context(OUTPUT)['settings']['training_seeds']:
            evaluate_condition(OUTPUT, method, seed, checkpoint_step=TARGET, stochastic=True)


## 保存結果の分析
途中結果も分析可能。実行済みepisode数を確認する。

In [ ]:
RUN_ANALYSIS = False
if RUN_ANALYSIS:
    report = analyze(OUTPUT, target=TARGET)
    from IPython.display import display, Markdown
    display(Markdown(report.read_text()))
